# Inf（全称 Infinity，即"无穷大"）
- 是 IEEE 754 浮点数标准中用于<font color='red'>表示超出可表示范围的数值的特殊值</font>。
- 它分为正无穷（+Inf）和负无穷（-Inf）。

## 核心特性
- 类型是 number：typeof Infinity 的结果是 "number"
- 比任何有限数都大：Infinity > 999999999999 → true
- 正负之分：+Infinity 和 -Infinity 是两个不同的值
- 运算规则明确：

In [ ]:
Infinity + 1        // Infinity
Infinity * 2        // Infinity
Infinity + Infinity // Infinity
-Infinity - 1       // -Infinity
1 / 0               // Infinity（注意不是 NaN）
-1 / 0              // -Infinity

## 与 NaN 的关键区别

| | Inf | NaN |
|---|---|---|
| 含义 | 数值太大，超出表示范围 | <font color='red'>运算结果无意义/未定义</font> |
| 等于自身 | `Infinity === Infinity` → `true` | `NaN === NaN` → `false` |
| 产生方式 | 除以零、数值溢出 | <font color='red'>0/0、无效运算、类型转换失败</font> |
| 运算结果 | 参与运算仍有确定规则 | 具有"传染性"，结果一律 NaN |

## 产生 Inf 的常见场景
### 除以零

In [ ]:
1 / 0       // Infinity
-1 / 0      // -Infinity

注意：非零数除以零是 Inf，不是 NaN。<font color='red'>只有 0 / 0 才是 NaN</font>。

### 数值溢出

In [ ]:
Number.MAX_VALUE * 2        // Infinity
Math.exp(1000)              // Infinity（e^1000 超出浮点数范围）

### <font color='red'>深度学习中的 Inf</font>
结合之前的讨论：
- 梯度爆炸：
  - 梯度值指数增长，超出 FP32 可表示范围（约 3.4 × 10^38）→ 变成 Inf
- 前向传播溢出：
  - exp(x) 中 x 过大 → 输出 Inf → loss 变成 Inf 或 NaN
- Inf 传播为 NaN：
  - Inf - Inf → NaN，Inf / Inf → NaN，0 * Inf → NaN
  
## Inf 如何变成 NaN
这是训练中最常见的"连环事故"：

In [ ]:
梯度爆炸 → 梯度变成 Inf
         → 权重更新后变成 Inf
         → 前向传播 exp(Inf) = Inf
         → loss 计算中 Inf - Inf = NaN
         → 后续所有梯度都是 NaN
         → 模型彻底无法训练

所以 Inf 往往是 NaN 的前兆。

## 如何检测

In [ ]:
# Python / PyTorch
import torch
torch.isinf(tensor)        # 检查是否有 Inf
torch.isfinite(tensor)     # 检查是否全部有限（既不是 Inf 也不是 NaN）

# JavaScript
Number.isFinite(x)         // 是否是有限数
Number.isInteger(x)        // 是否是整数

## 在 IEEE 754 中的底层表示
以 32 位单精度浮点数为例：

| 值 | 符号位 | 指数位 | 尾数位 |
|---|---|---|---|
| +Infinity | 0 | 全 1（255） | 全 0 |
| -Infinity | 1 | 全 1（255） | 全 0 |
| NaN | 任意 | 全 1（255） | 不全为 0 |

可以看到 Inf 和 NaN 的指数位都是全 1，区别在于尾数位：全 0 是 Inf，非全 0 是 NaN。

## 一句话总结
- Inf 是浮点数系统中"数值太大装不下"的标记。它本身有确定的运算规则，
  - <font color='red'>但一旦 Inf 参与 Inf - Inf、Inf / Inf、0 × Inf 这类"无穷对无穷"的运算，就会退化为 NaN</font>
  - <font color='blue'>这也是训练中 loss 从 Inf 变成 NaN 的根本原因</font>。